<a href="https://colab.research.google.com/github/aurorafinetti/progetto.tecnologiedeidatiedellinguaggio/blob/main/Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests pandas ipywidgets

import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from collections import defaultdict
import time

# ── CONFIGURAZIONE ────────────────────────────────────────────────────
TOKEN = "YmYWIbBkQVihZyWWxXIAPhvMZjSeWeRGSxPGOZnx"

headers = {
    "Authorization": f"Discogs token={TOKEN}",
    "User-Agent": "ProgettoVinili/1.0"
}

# ── RACCOLTA DATI DA DISCOGS ──────────────────────────────────────────
print("🔍 Recupero dati da Discogs...")

dati = []
for pagina in range(1, 6):
    print(f"  Scarico pagina {pagina}/5...")
    url = "https://api.discogs.com/database/search"
    params = {
        "type": "release",
        "format": "Vinyl",
        "country": "US",
        "year": 2025,
        "sort": "have",
        "sort_order": "desc",
        "per_page": 50,
        "page": pagina
    }
    response = requests.get(url, headers=headers, params=params)
    risultati = response.json().get("results", [])

    for r in risultati:
        dati.append({
            "titolo": r.get("title", "N/A"),
            "anno": r.get("year", "N/A"),
            "genere": ", ".join(r.get("genre", [])),
            "stile": ", ".join(r.get("style", [])),
            "etichetta": ", ".join(r.get("label", [])),
            "hanno": r.get("community", {}).get("have", 0),
            "vogliono": r.get("community", {}).get("want", 0),
        })
    time.sleep(1)

df = pd.DataFrame(dati)
df = df.sort_values("hanno", ascending=False).reset_index(drop=True)
print(f"✅ Trovati {len(df)} vinili.\n")

# Salva in CSV
df.to_csv("discogs_vinili_2025.csv", index=False)

# ── TABELLA COMPLETA ──────────────────────────────────────────────────
def genera_tabella(dataframe, titolo=""):
    righe_html = ""
    colore_alt = False
    for _, r in dataframe.iterrows():
        bg = "#f9f9f9" if colore_alt else "#ffffff"
        colore_alt = not colore_alt
        righe_html += (
            f"<tr style='background:{bg}'>"
            f"<td style='padding:5px 12px; text-align:center'><b>{r['anno']}</b></td>"
            f"<td style='padding:5px 12px'>{r['titolo']}</td>"
            f"<td style='padding:5px 12px'>{r['genere']}</td>"
            f"<td style='padding:5px 12px'>{r['stile']}</td>"
            f"<td style='padding:5px 12px; text-align:center'>{r['hanno']}</td>"
            f"<td style='padding:5px 12px; text-align:center'>{r['vogliono']}</td>"
            f"</tr>"
        )
    return HTML(f"""
    <h3>🎵 {titolo}</h3>
    <p style='color:gray'>Totale: <b>{len(dataframe)}</b> vinili</p>
    <div style='max-height:500px; overflow-y:auto'>
    <table border='1' cellspacing='0' style='border-collapse:collapse; font-size:13px; width:100%'>
      <thead style='background:#1a1a2e; color:white; position:sticky; top:0'>
        <tr>
          <th style='padding:7px 12px'>Anno</th>
          <th style='padding:7px 12px'>Titolo</th>
          <th style='padding:7px 12px'>Genere</th>
          <th style='padding:7px 12px'>Stile</th>
          <th style='padding:7px 12px'>Hanno</th>
          <th style='padding:7px 12px'>Vogliono</th>
        </tr>
      </thead>
      <tbody>{righe_html}</tbody>
    </table>
    </div>
    """)

display(genera_tabella(df, "Vinili più popolari su Discogs – USA 2025"))

# ── CONTEGGIO PER GENERE ──────────────────────────────────────────────
conteggio_genere = defaultdict(int)
for genere in df["genere"]:
    for g in genere.split(", "):
        if g and g != "N/A":
            conteggio_genere[g] += 1

righe_genere = "".join(
    f"<tr><td style='padding:4px 16px'><b>{g}</b></td>"
    f"<td style='padding:4px 16px; text-align:center'>{'🎵' * min(n, 10)} {n}</td></tr>"
    for g, n in sorted(conteggio_genere.items(), key=lambda x: -x[1])
)

display(HTML(f"""
<h3>📊 Distribuzione per genere</h3>
<table border='1' cellspacing='0' style='border-collapse:collapse; font-size:13px'>
  <thead style='background:#1a1a2e; color:white'>
    <tr>
      <th style='padding:7px 16px'>Genere</th>
      <th style='padding:7px 16px'>N° vinili</th>
    </tr>
  </thead>
  <tbody>{righe_genere}</tbody>
</table>
"""))

# ── RICERCA INTERATTIVA ───────────────────────────────────────────────
testo_input = widgets.Text(
    value="Rock",
    description="🔎 Cerca:",
    placeholder="Anno (es. 2025), genere (es. Rock) o titolo",
    layout=widgets.Layout(width="420px")
)
btn = widgets.Button(description="Cerca", button_style="primary", icon="search")
output = widgets.Output()

def cerca(b):
    output.clear_output()
    query = testo_input.value.strip()
    with output:
        if not query:
            print("⚠️ Inserisci un anno, un genere o un titolo.")
            return

        if query.isdigit():
            risultati = df[df["anno"] == query]
        else:
            q = query.lower()
            risultati = df[
                df["titolo"].str.lower().str.contains(q, na=False) |
                df["genere"].str.lower().str.contains(q, na=False) |
                df["stile"].str.lower().str.contains(q, na=False)
            ]

        if risultati.empty:
            display(HTML(f"<span style='color:red'>❌ Nessun risultato per '<b>{query}</b>'.</span>"))
            return

        display(genera_tabella(risultati, f"Risultati per \"{query}\""))

btn.on_click(cerca)
display(HTML("<h3>🔎 Filtra per anno, genere o titolo</h3>"))
display(widgets.HBox([testo_input, btn]), output)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 22.2 MB/s eta 0:00:00
🔍 Recupero dati da Discogs...
  Scarico pagina 1/5...
  Scarico pagina 2/5...
  Scarico pagina 3/5...
  Scarico pagina 4/5...
  Scarico pagina 5/5...
✅ Trovati 250 vinili.



Anno,Titolo,Genere,Stile,Hanno,Vogliono
2025,Taylor Swift - The Life Of A Showgirl,"Electronic, Pop","Synth-pop, Ballad, Downtempo, Soft Rock, Dance-pop",28194,1141
2025,Taylor Swift - The Life Of A Showgirl,"Electronic, Pop","Synth-pop, Ballad, Downtempo, Soft Rock, Dance-pop",18071,731
2025,Frank Ocean - Channel Orange,"Hip Hop, Funk / Soul, Pop","Contemporary R&B, Soul",14643,6636
2025,Taylor Swift Featuring Post Malone - Fortnight,"Electronic, Pop","Synth-pop, Indie Pop, Dance-pop",11739,857
2025,Geese (11) - Getting Killed,Rock,"Folk Rock, Experimental, Indie Rock, Alternative Rock",10901,3243
2025,Sabrina Carpenter - Short N' Sweet (Deluxe),"Funk / Soul, Pop","Bubblegum, Contemporary R&B, Nu-Disco, Vocal",10651,2052
2025,Ariana Grande - Eternal Sunshine Deluxe: Brighter Days Ahead,Pop,"Contemporary R&B, Trap, Dance-pop",6859,1305
2025,Sabrina Carpenter - Man's Best Friend,"Pop, Folk, World, & Country","Country, Indie Pop",6604,601
2025,Cindy Lee - Diamond Jubilee,"Rock, Pop","Hypnagogic pop, Indie Rock, Psychedelic Rock",6437,1509
2025,Kendrick Lamar - GNX,"Hip Hop, Funk / Soul","Conscious, Contemporary R&B, Hardcore Hip-Hop",6381,884


Genere,N° vinili
Pop,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 117
Rock,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 105
Hip Hop,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 50
Electronic,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 49
Funk / Soul,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 34
Folk,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 22
World,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 22
& Country,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 22
Stage & Screen,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 14
Jazz,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 13


Output()

In [ ]:
!pip install requests pandas ipywidgets musicbrainzngs

import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from collections import defaultdict
import musicbrainzngs
import time

# ── CONFIGURAZIONE ────────────────────────────────────────────────────
musicbrainzngs.set_useragent("ProgettoVinili", "1.0", "aurora.1193@libero.it")

# ── RACCOLTA DATI DA MUSICBRAINZ ──────────────────────────────────────
print("🔍 Recupero dati da MusicBrainz...")

# Lista di artisti da cercare — puoi modificarla
artisti_da_cercare = [
    "Taylor Swift", "Sabrina Carpenter", "Billie Eilish", "Kendrick Lamar",
    "Morgan Wallen", "SZA", "Bad Bunny", "The Weeknd", "Oasis", "Fleetwood Mac",
    "Sam Fender", "Fontaines D.C.", "Olivia Dean", "Chappell Roan", "Beyoncé",
    "Harry Styles", "Arctic Monkeys", "Radiohead", "Frank Ocean", "Tyler the Creator"
]

dati = []

for artista in artisti_da_cercare:
    print(f"  Cerco: {artista}...")
    try:
        # Cerca l'artista
        risultato = musicbrainzngs.search_artists(artist=artista, limit=1)
        if not risultato["artist-list"]:
            continue

        artista_mb = risultato["artist-list"][0]
        artista_id = artista_mb["id"]
        artista_nome = artista_mb["name"]

        # Cerca i tag (generi) dell'artista
        dettagli = musicbrainzngs.get_artist_by_id(artista_id, includes=["tags"])
        tags = dettagli["artist"].get("tag-list", [])
        generi = ", ".join([t["name"] for t in tags[:3]])  # prende i primi 3 tag

        # Cerca gli album dell'artista
        release_groups = musicbrainzngs.search_release_groups(
            artistname=artista_nome,
            type="album",
            limit=5
        )

        for album in release_groups["release-group-list"]:
            dati.append({
                "artista": artista_nome,
                "album": album.get("title", "N/A"),
                "anno": album.get("first-release-date", "N/A")[:4] if album.get("first-release-date") else "N/A",
                "genere": generi if generi else "N/A"
            })

        time.sleep(1)

    except Exception as e:
        print(f"  ⚠️ Errore per {artista}: {e}")
        continue

df = pd.DataFrame(dati)
df = df.reset_index(drop=True)
print(f"✅ Trovati {len(df)} album.\n")

# Salva in CSV
df.to_csv("musicbrainz_generi.csv", index=False)

# ── TABELLA COMPLETA ──────────────────────────────────────────────────
def genera_tabella(dataframe, titolo=""):
    righe_html = ""
    colore_alt = False
    for _, r in dataframe.iterrows():
        bg = "#f9f9f9" if colore_alt else "#ffffff"
        colore_alt = not colore_alt
        righe_html += (
            f"<tr style='background:{bg}'>"
            f"<td style='padding:5px 12px'>{r['artista']}</td>"
            f"<td style='padding:5px 12px'>{r['album']}</td>"
            f"<td style='padding:5px 12px; text-align:center'>{r['anno']}</td>"
            f"<td style='padding:5px 12px'>{r['genere']}</td>"
            f"</tr>"
        )
    return HTML(f"""
    <h3>🎵 {titolo}</h3>
    <p style='color:gray'>Totale: <b>{len(dataframe)}</b> album</p>
    <div style='max-height:500px; overflow-y:auto'>
    <table border='1' cellspacing='0' style='border-collapse:collapse; font-size:13px; width:100%'>
      <thead style='background:#eb743b; color:white; position:sticky; top:0'>
        <tr>
          <th style='padding:7px 12px'>Artista</th>
          <th style='padding:7px 12px'>Album</th>
          <th style='padding:7px 12px'>Anno</th>
          <th style='padding:7px 12px'>Genere</th>
        </tr>
      </thead>
      <tbody>{righe_html}</tbody>
    </table>
    </div>
    """)

display(genera_tabella(df, "MusicBrainz – Generi musicali per artista"))

# ── CONTEGGIO PER GENERE ──────────────────────────────────────────────
conteggio_genere = defaultdict(int)
for genere in df["genere"]:
    for g in genere.split(", "):
        if g and g != "N/A":
            conteggio_genere[g] += 1

righe_genere = "".join(
    f"<tr><td style='padding:4px 16px'><b>{g}</b></td>"
    f"<td style='padding:4px 16px; text-align:center'>{'🎵' * min(n, 10)} {n}</td></tr>"
    for g, n in sorted(conteggio_genere.items(), key=lambda x: -x[1])
)

display(HTML(f"""
<h3>📊 Distribuzione per genere</h3>
<table border='1' cellspacing='0' style='border-collapse:collapse; font-size:13px'>
  <thead style='background:#eb743b; color:white'>
    <tr>
      <th style='padding:7px 16px'>Genere</th>
      <th style='padding:7px 16px'>N° album</th>
    </tr>
  </thead>
  <tbody>{righe_genere}</tbody>
</table>
"""))

# ── RICERCA INTERATTIVA ───────────────────────────────────────────────
testo_input = widgets.Text(
    value="pop",
    description="🔎 Cerca:",
    placeholder="Artista, album o genere",
    layout=widgets.Layout(width="420px")
)
btn = widgets.Button(description="Cerca", button_style="primary", icon="search")
output = widgets.Output()

def cerca(b):
    output.clear_output()
    query = testo_input.value.strip()
    with output:
        if not query:
            print("⚠️ Inserisci un artista, album o genere.")
            return

        q = query.lower()
        risultati = df[
            df["artista"].str.lower().str.contains(q, na=False) |
            df["album"].str.lower().str.contains(q, na=False) |
            df["genere"].str.lower().str.contains(q, na=False)
        ]

        if risultati.empty:
            display(HTML(f"<span style='color:red'>❌ Nessun risultato per '<b>{query}</b>'.</span>"))
            return

        display(genera_tabella(risultati, f"Risultati per \"{query}\""))

btn.on_click(cerca)
display(HTML("<h3>🔎 Filtra per artista, album o genere</h3>"))
display(widgets.HBox([testo_input, btn]), output)

🔍 Recupero dati da MusicBrainz...
  Cerco: Taylor Swift...
  Cerco: Sabrina Carpenter...
  Cerco: Billie Eilish...
  Cerco: Kendrick Lamar...
  Cerco: Morgan Wallen...
  Cerco: SZA...
  Cerco: Bad Bunny...
  Cerco: The Weeknd...
  Cerco: Oasis...
  Cerco: Fleetwood Mac...
  Cerco: Sam Fender...
  Cerco: Fontaines D.C....
  Cerco: Olivia Dean...
  Cerco: Chappell Roan...
  Cerco: Beyoncé...
  Cerco: Harry Styles...
  Cerco: Arctic Monkeys...
  Cerco: Radiohead...
  Cerco: Frank Ocean...
  Cerco: Tyler the Creator...
✅ Trovati 100 album.



Artista,Album,Anno,Genere
Taylor Swift,The Taylor Swift Megamix,2015,"2000s, 2010s, 2020s"
Taylor Swift,Taylor Swift Karaoke: Speak Now,2010,"2000s, 2010s, 2020s"
Taylor Swift,Speak Now: World Tour Live,2011,"2000s, 2010s, 2020s"
Taylor Swift,Taylor Swift Karaoke: Fearless,2009,"2000s, 2010s, 2020s"
Taylor Swift,Hot Champion,2016,"2000s, 2010s, 2020s"
Sabrina Carpenter,Singular: Act III - The Sessions,2024,"2020s, american, dance-pop"
Sabrina Carpenter,emails i can't send (demos),2024,"2020s, american, dance-pop"
Sabrina Carpenter,dream girl,2024,"2020s, american, dance-pop"
Sabrina Carpenter,Track by Track: Sabrina Carpenter,2024,"2020s, american, dance-pop"
Sabrina Carpenter,Singular: Act II,2019,"2020s, american, dance-pop"


Genere,N° album
2020s,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 40
2010s,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 25
alternative rock,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 20
2000s,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 10
american,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 10
dance-pop,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 10
alternative r&b,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 10
hip hop,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 10
big band,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 10
alternative,🎵🎵🎵🎵🎵🎵🎵🎵🎵🎵 10


Output()